# 01 · Transaction data understanding
**Company:** *Tausi Digital Bank*, a **fictional** Kenyan bank. All customer, account and transaction data is **synthetic** (`src/generate_data.py`). It does not describe any real bank, mobile-money provider or customer.

**Why synthetic?** I reviewed PaySim (Lopez-Rojas et al., 2016, CC BY-SA 4.0), the best-known public mobile-money dataset. It is itself simulated, covers only 30 days, and has no channel, status, failure reason, fee or customer fields. It was also built for fraud research, and this project is deliberately **not** about fraud: it is about volume, channels, reliability, fees and customer activity.

**The one real dataset** here is the Central Bank of Kenya's monthly mobile-payments statistics (`data/context/`). They are used only as a market trend line and never mixed into the synthetic data.

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data_loader import get_engine, query, PROJECT_ROOT
from src import analysis as an
an.set_style()
pd.set_option("display.float_format", "{:,.2f}".format); pd.set_option("display.max_columns", 30)
engine = get_engine()
o = an.sql_output

In [2]:
query(engine, """
SELECT 'customers' t, COUNT(*) n, 'one customer' grain FROM bank.customers
UNION ALL SELECT 'accounts', COUNT(*), 'one account (each customer has one primary account)' FROM bank.accounts
UNION ALL SELECT 'transactions', COUNT(*), 'one attempt: successful or failed; a retry is a new row' FROM bank.transactions
UNION ALL SELECT 'quarantined_transactions', COUNT(*), 'raw rows held back by cleaning, with a reason' FROM bank.quarantined_transactions
UNION ALL SELECT 'fee_schedule', COUNT(*), 'one tariff band: type x channel x amount band x validity period' FROM bank.fee_schedule
UNION ALL SELECT 'system_incidents', COUNT(*), 'one logged outage' FROM bank.system_incidents
UNION ALL SELECT 'context_cbk_mobile_payments', COUNT(*), 'REAL national statistics, one month' FROM bank.context_cbk_mobile_payments
UNION ALL SELECT 'mv_customer_month', COUNT(*), 'derived: one row per onboarded customer per month' FROM bank.mv_customer_month""")

,t,n,grain
0,customers,30000,one customer
1,accounts,38034,one account (each customer has one primary acc...
2,transactions,2022688,one attempt: successful or failed; a retry is ...
3,quarantined_transactions,130,"raw rows held back by cleaning, with a reason"
4,fee_schedule,65,one tariff band: type x channel x amount band ...
5,system_incidents,11,one logged outage
6,context_cbk_mobile_payments,24,"REAL national statistics, one month"
7,mv_customer_month,430738,derived: one row per onboarded customer per month


## Key definitions (from `sql/03_views.sql`)
| Term | Definition |
|---|---|
| Attempt | one row in `bank.transactions`; a retry is a new attempt |
| Success rate | successful attempts ÷ all attempts |
| Operational failure | failure owned by the bank or a partner: timeout, USSD session dropped, agent float, third-party unavailable, outage |
| Customer failure | insufficient funds, wrong PIN, limit exceeded; reported separately, not a reliability problem |
| **Transaction value** | KES moved by successful attempts. **Not revenue** |
| **Fee income** | fees charged on successful attempts: the bank's transaction revenue |
| Retry | next attempt on the same account, same type and amount, within 30 minutes |
| Active customer | at least one successful transaction in the month |
| Dormant | no successful transaction in the 90 days to 31 Oct 2026 |

In [3]:
query(engine, """SELECT channel, transaction_type, COUNT(*) attempts, ROUND(AVG(amount)) avg_amount_kes,
       ROUND(100.0*AVG(is_success::int),1) success_pct FROM bank.mv_transactions GROUP BY 1,2 ORDER BY 1,3 DESC""")

,channel,transaction_type,attempts,avg_amount_kes,success_pct
0,ATM,Cash Withdrawal,226442,"6,887.00",97.50
1,Agent,Cash Withdrawal,132845,"8,345.00",93.60
2,Agent,Cash Deposit,106608,"11,011.00",98.90
3,Internet Banking,Interbank Transfer,105313,"117,312.00",96.10
4,Internet Banking,Internal Transfer,70030,"35,928.00",98.40
5,Internet Banking,Bill Payment,58428,"11,011.00",97.50
6,Mobile App,Send to Mobile Wallet,267525,"4,100.00",96.50
7,Mobile App,Merchant Payment,137795,"2,815.00",97.40
8,Mobile App,Airtime Purchase,137650,257.00,97.30
9,Mobile App,Bill Payment,121271,"5,318.00",96.30


## One customer, end to end
A Salaried customer's account history shows the grain: attempts, a failed attempt followed by a retry, fees only on successful fee-bearing transactions.

In [4]:
cid = query(engine, """SELECT customer_id FROM bank.mv_transactions WHERE segment='Salaried' AND was_retried
                         AND month = DATE '2026-09-01' ORDER BY transaction_id LIMIT 1""").iloc[0, 0]
query(engine, f"""SELECT txn_ts, channel, transaction_type, amount, status, failure_reason, fee_charged, was_retried, retry_status
                    FROM bank.mv_transactions WHERE customer_id = '{cid}' AND month = DATE '2026-09-01' ORDER BY txn_ts""")

,txn_ts,channel,transaction_type,amount,status,failure_reason,fee_charged,was_retried,retry_status
0,2026-09-01 09:50:04,USSD,Send to Mobile Wallet,"1,249.00",Success,NaN,28.00,False,NaN
1,2026-09-01 12:07:55,Mobile App,Send to Mobile Wallet,239.00,Failed,Third-party unavailable,0.00,True,Success
2,2026-09-01 12:09:25,Mobile App,Send to Mobile Wallet,239.00,Success,NaN,0.00,False,NaN
3,2026-09-03 21:30:35,Mobile App,Merchant Payment,"1,468.00",Success,NaN,0.00,False,NaN
4,2026-09-04 13:34:43,Mobile App,Internal Transfer,"4,638.00",Success,NaN,0.00,False,NaN
5,2026-09-05 05:08:07,Mobile App,Interbank Transfer,"36,745.00",Success,NaN,100.00,False,NaN
6,2026-09-05 14:01:54,Wallet Link,Wallet Top-up,"2,865.00",Success,NaN,0.00,False,NaN
7,2026-09-09 17:43:58,Mobile App,Send to Mobile Wallet,"3,281.00",Success,NaN,25.00,False,NaN
8,2026-09-11 14:56:06,Agent,Cash Withdrawal,"1,580.00",Success,NaN,60.00,False,NaN
9,2026-09-13 03:08:17,ATM,Cash Withdrawal,500.00,Success,NaN,33.00,False,NaN


In [5]:
query(engine, "SELECT segment, COUNT(*) customers, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),1) pct, MIN(onboarding_date) first_onboarded FROM bank.customers GROUP BY 1 ORDER BY 2 DESC")

,segment,customers,pct,first_onboarded
0,Mass Retail,11952,39.80,2021-05-03
1,Youth,6575,21.90,2021-05-03
2,Salaried,6002,20.00,2021-05-03
3,SME,3632,12.10,2021-05-03
4,Premier,1839,6.10,2021-05-05


In [6]:
query(engine, 'SELECT * FROM bank.fee_schedule ORDER BY transaction_type, channel, band_min_kes, effective_from').head(20)

,tariff_id,transaction_type,channel,band_min_kes,band_max_kes,fee_kes,effective_from,effective_to
0,F059,Bill Payment,Internet Banking,1.00,100.00,15.00,2025-01-01,None
1,F060,Bill Payment,Internet Banking,101.00,"1,000.00",15.00,2025-01-01,None
2,F061,Bill Payment,Internet Banking,"1,001.00","5,000.00",15.00,2025-01-01,None
3,F062,Bill Payment,Internet Banking,"5,001.00","20,000.00",15.00,2025-01-01,None
4,F063,Bill Payment,Internet Banking,"20,001.00","70,000.00",15.00,2025-01-01,None
5,F064,Bill Payment,Internet Banking,"70,001.00","250,000.00",15.00,2025-01-01,None
6,F065,Bill Payment,Internet Banking,"250,001.00","999,999,999.00",15.00,2025-01-01,None
7,F045,Bill Payment,Mobile App,1.00,100.00,15.00,2025-01-01,None
8,F046,Bill Payment,Mobile App,101.00,"1,000.00",15.00,2025-01-01,None
9,F047,Bill Payment,Mobile App,"1,001.00","5,000.00",15.00,2025-01-01,None
